# Formattage des données de paludisme hebdomadaires (DSE)

**Auteur** : Alex Kaldjian

**Date** : Janvier 2022

**Projet** : MACEPA

**Pays** : RDC

## Chargement des librairies et connexion à la base de données PostGres

In [ ]:
options(repr.matrix.max.cols=30,
        repr.plot.width=12, 
        repr.plot.height=6)

In [ ]:
install.packages("scales", quiet = TRUE) ## install this extra library
install.packages("zoo", quiet = TRUE)

In [ ]:
library(data.table)
library(ggplot2)
library(RPostgres) 
library(sf)
library(zoo)
library(stringr)

In [ ]:
con <- dbConnect(
    RPostgres::Postgres(),
    dbname = Sys.getenv("WORKSPACE_DATABASE_DB_NAME"),
    host = Sys.getenv("WORKSPACE_DATABASE_HOST"),
    port = Sys.getenv("WORKSPACE_DATABASE_PORT"),
    user = Sys.getenv("WORKSPACE_DATABASE_USERNAME"),
    password = Sys.getenv("WORKSPACE_DATABASE_PASSWORD")
)

In [ ]:
# Helper functions
corriger_ZS <- function(nom, df) {
    # find IDs of all observations with given zone name
    id = unique(df[ZS == nom, c(ID)])
    
    for(i in id) {
            # find names of all zones with that ID
            noms_possibles = df[ID == i, 
                                .N, 
                                by = ZS][order(-N), c(ZS)]
            
            # chose name with most observations
            nom_choisi = noms_possibles[1]

            # assign correct name to all relevant observations 
            df[ZS == nom & ID == i, ZS := nom_choisi]
    }
    
    return()
}

suma <- function(x) {
            if (all(is.na(x))) {
                    return(x[NA_integer_])
            } else {
                return(sum(x, na.rm = TRUE))
            }
        }

extract_year_from_path <- function(path) {
  file_name <- basename(path)
  # Check if filename starts with 4 digits
  if (!grepl("^[0-9]{4}", file_name)) {
    return(NULL)
  }
  
  year <- as.numeric(sub("^([0-9]{4}).*", "\\1", file_name))
  
  # Extra safety check (in case conversion fails)
  if (is.na(year)) {
    return(NULL)
  }  
  return(year)
}

# Function to get the latest week file from a folder
get_latest_week_file <- function(folder_path, pattern = "_Week([0-9]+)\\.csv$") {
  all_files <- list.files(folder_path, full.names = TRUE)
  
  # Filter files matching the week pattern
  week_files <- all_files[str_detect(all_files, pattern)]
  
  # If no files found, return NULL
  if(length(week_files) == 0){
    warning("No files matching the pattern found in folder: ", folder_path)
    return(NULL)
  }
  
  # Extract week numbers
  week_numbers <- str_extract(week_files, pattern) %>% 
    str_extract("[0-9]+") %>% 
    as.integer()
  
  # Return the file with the max week number
  week_files[which.max(week_numbers)]
}

## 1. Chargement des données pour calculs des outbreaks et cleaning des BD

Path vers le nouveau fichier d'input ici:

In [ ]:
# nouvelles_donnees_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2024/2024_Database_Week02.csv"
# nouvelles_donnees_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2025/2025_Database_Week18.csv"
# nouvelles_donnees_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2026/2026_Database_Week05.csv"

In [ ]:
base_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/"
raw_data_path <- paste0(base_path, "raw/")
metadata_path <- paste0(base_path, "metadata/")
output_path <- paste0(base_path, "outputs/")
historic_path <- file.path(base_path, "raw", "historic")

In [ ]:
## NEW CODE

### 1.1 Load current year data

In [ ]:
# Retrieve year from input filepath
current_year <- extract_year_from_path(nouvelles_donnees_path)
if (is.null(current_year)) {
    stop("Impossible de trouver l’année de référence dans le fichier. Le processus est arrêté.")
}
current_year

In [ ]:
# fix to multiple column length error 
dse_current <- fread(nouvelles_donnees_path, sep = ";", encoding = "Latin-1")  # or "UTF-8"

# Handle comma separated file format (usually saved after manual checking)
if (length(colnames(dse_current)) == 1) {
    dse_current <- fread(nouvelles_donnees_path, sep = ",", encoding = "Latin-1")  # or "UTF-8"
}
setDT(dse_current)

# Convert only character columns safely to UTF-8
# dse_current[, (names(dse_current)) := lapply(.SD, function(x) {
    # if (is.character(x)) {iconv(x, from = "Latin1", to = "UTF-8", sub = "")} else {x}})]

print(colnames(dse_current)) 
# print(glue::glue("Invalid characters: {any(!validUTF8(unlist(dse_current)))}"))

In [ ]:
dse_current[, C5ANSP := rowSums(.SD, na.rm=T), .SDcols = c('C515ANS', 'CP15ANS')]
dse_current[, D5ANSP := rowSums(.SD, na.rm=T), .SDcols = c('D515ANS', 'DP15ANS')]
dse_current[, C515ANS := NULL][, D515ANS := NULL][, CP15ANS := NULL][, DP15ANS := NULL]
dse_current[, year := current_year]  # UPDATE YEAR

dse_current[, TOTALDECES := as.integer(TOTALDECES)]
dse_current[is.na(TOTALDECES), TOTALCAS := rowSums(.SD, na.rm=T), .SDcols = c('C011MOIS', 'C1259MOIS', 'C5ANSP')]
dse_current[is.na(TOTALDECES), TOTALDECES := rowSums(.SD, na.rm=T), .SDcols = c('D011MOIS', 'D1259MOIS', 'D5ANSP')]
dse_current[, MALADIE := toupper(MALADIE)]

# 2020- PNLP a dédcidé d'utiliser PALU SUSP
# dse_current[MALADIE == 'PALUDISME SUSP', MALADIE := 'PALUDISME']
# dse_current <- dse_current[MALADIE != 'PALUDISME CONF']

# 2025- PNLP a dédcidé d'utiliser PALU CONF
dse_current[MALADIE == 'PALUDISME CONF', MALADIE := 'PALUDISME']
dse_current <- dse_current[MALADIE != 'PALUDISME SUSP']
head(dse_current,3)

### 1.2 Load previous years data

In [ ]:
# Load DSE folder data
all_dirs <- list.dirs(raw_data_path, full.names = TRUE, recursive = FALSE)
dse_folders <- all_dirs[grepl("DSE_data_", basename(all_dirs))]
dse_folders <- dse_folders[!grepl(as.character(current_year), basename(dse_folders))] # except current year folder

# Load last available file from each folder
dse_prev_years <- data.table()
for (dse_folder in dse_folders) {

    dse_filename <- get_latest_week_file(dse_folder)
    file_year <- extract_year_from_path(dse_filename)
    print(glue::glue("Chargement des données pour l’ancienne année {file_year}: {dse_filename}"))
    
    dse_data_year <-  fread(dse_filename, sep = ";", encoding = "Latin-1", quote = "")  # or "UTF-8"
    # Handle comma separated file format (usually saved after manual checking)
    if (length(colnames(dse_data_year)) < 3) {         
        dse_data_year <-  fread(dse_filename, sep = ",", encoding = "Latin-1", quote = "")  # or "UTF-8" 
    }
    
    setDT(dse_data_year)
    dse_data_year[, year := file_year]
    dse_data_year[, C5ANSP := rowSums(.SD, na.rm=T), .SDcols = c('C515ANS', 'CP15ANS')]
    dse_data_year[, D5ANSP := rowSums(.SD, na.rm=T), .SDcols = c('D515ANS', 'DP15ANS')]
    dse_data_year[, C515ANS := NULL][, D515ANS := NULL][, CP15ANS := NULL][, DP15ANS := NULL]
    
    if (file_year %in% c(2022, 2023, 2024, 2025)) {        
        dse_data_year[, TOTALDECES := as.integer(TOTALDECES)]
        dse_data_year[is.na(TOTALDECES), TOTALCAS := rowSums(.SD, na.rm=T), .SDcols = c('C011MOIS', 'C1259MOIS', 'C5ANSP')]
        dse_data_year[is.na(TOTALDECES), TOTALDECES := rowSums(.SD, na.rm=T), .SDcols = c('D011MOIS', 'D1259MOIS', 'D5ANSP')]        
        dse_data_year[, MALADIE := toupper(MALADIE)]
        
        # 2020 PNLP a décidé d'utiliser PALU SUSP
        # dse_data_year[MALADIE == 'PALUDISME SUSP', MALADIE := 'PALUDISME']
        # dse_data_year <- dse_data_year[MALADIE != 'PALUDISME CONF']
        
        # 2025 PNLP a décidé d'utiliser PALU CONF
        dse_data_year[MALADIE == 'PALUDISME CONF', MALADIE := 'PALUDISME']
        dse_data_year <- dse_data_year[MALADIE != 'PALUDISME SUSP']
    }
    dse_prev_years <- rbind(dse_prev_years, dse_data_year, fill = TRUE)
}

In [ ]:
unique(dse_prev_years$year)

### 1.3 Load historic data

In [ ]:
# Load DSE historic data
dse_hits_files <- list.files(historic_path, full.names=TRUE)
dse_hits_files <- dse_hits_files[!grepl("2016|2017|2018", dse_hits_files)]  # Exclude old files (legacy code)

dse_hist <- data.table()
for (dse_hist_file in dse_hits_files) {    
    file_year <- extract_year_from_path(dse_hist_file)
    
    print(glue::glue("Chargement des données historiques pour l’année {file_year}: {dse_hist_file}"))
    dse_hist_year <- fread(dse_hist_file, sep = ";")    
    dse_hist_year[, year := file_year]

    if (file_year == 2020) {
       dse_hist_year[DEBUTSEM == "28/12/2021", DEBUTSEM := "28/12/2020"] 
    }
    if (file_year == 2018) {
        dse_hist_year[NUMSEM == 0, NUMSEM := 1]
    }
    
    dse_hist <- rbind(dse_hist, dse_hist_year, fill = TRUE)
}

In [ ]:
unique(dse_hist$year)

In [ ]:
# Concat data
combo <- rbind(dse_hist, dse_prev_years, dse_current, fill = TRUE)

In [ ]:
dim(combo)
head(combo, 3)

In [ ]:
# Check all years are listed here
unique(combo$year)

In [ ]:
## NEW CODE

## Nettoyage des données

In [ ]:
# all sorts of different province names
sort(unique(combo$PROV))
length(sort(unique(combo$PROV)))

In [ ]:
# fix province names + drop empties
combo[, PROV := gsub("-", " ", PROV)]
combo[PROV == "LULABA", PROV := "LUALABA"]
combo <- combo[PROV != ""]

# drop NA weeks
combo <- combo[!is.na(NUMSEM)]
combo <- combo[NUMSEM != 0]
combo <- combo[NUMSEM < 53]

# reduce to necessary columns
combo <- combo[, .(
  NUM, PROV, ZS, NUMSEM, MALADIE, C011MOIS, D011MOIS,
  C1259MOIS, D1259MOIS, C5ANSP, D5ANSP, TOTALCAS, TOTALDECES,
  year
)]

In [ ]:
sort(unique(combo$PROV))
tot_diff_prov <- length(sort(unique(combo$PROV)))
print(tot_diff_prov)
if (tot_diff_prov != 26) {
    stop("Number of unique province names in the dataset is not 26: {}. Process halted.")
}

In [ ]:
combo[, NUM := iconv(NUM, from = "", to = "UTF-8", sub = "byte")] # Force R to convert all values in NUM to UTF-8 and replaces invalid characters
combo[, ID := gsub("[[:digit:]]", "", NUM)]

noms_a_corriger <- combo[, .N, by = .(ZS, PROV)][N < 250, c(ZS)]

invisible(lapply(noms_a_corriger, corriger_ZS, combo))

In [ ]:
combo[, .N, by = .(PROV, ZS)][N < 200]

In [ ]:
# fix the last few, then drop the rest
combo <- merge(combo,
  combo[, .N, by = .(PROV, ZS)],
  all.x = T
)

combo[N < 200, ZS := gsub(" ", "", ZS)]

combo <- combo[N > 100]

In [ ]:
all_names <- combo[, .N, by = .(PROV, ZS)][order(N)]

## Import zone de santé dictionary from love machine

In [ ]:
zs_dict <- fread(
    paste0(metadata_path, "love_machine_zone_dict.csv"),
    header = TRUE
)

In [ ]:
zs_dict[which(zs_dict$zone_ref == 'nu Bili Zone de Santé'), ]

In [ ]:
combo[which(combo$ZS == 'BILI')]

In [ ]:
zs_dict <- zs_dict[, .(
    ZS = zone_DSE, PROV = province_DSE,
    Nom = zone_ref, PROVINCE = province_ref,
    zone_id_DHIS2, province_id_DHIS2
)]

In [ ]:
head(zs_dict)

In [ ]:
zs_dict[which(zs_dict$Nom == 'nu Bili Zone de Santé'), ]

In [ ]:
test_allx <- merge(all_names, zs_dict, all.x = T, by = c("ZS", "PROV"))

Have to fix a few manually...

In [ ]:
test_allx[is.na(Nom)]

In [ ]:
manually_corrected <- fread(
    paste0(metadata_path, "manually_corrected_full.csv"),
    header = TRUE
)

manually_corrected <- manually_corrected[, .(
    ZS = zone_DSE, PROV = province_DSE,
    Nom = zone_ref, PROVINCE = province_ref,
    zone_id_DHIS2, province_id_DHIS2
)]

In [ ]:
zs_dict <- rbind(zs_dict[!(ZS %in% manually_corrected$ZS)], manually_corrected)

In [ ]:
test_allx <- merge(all_names, zs_dict, all.x = T, by = c("ZS", "PROV"))

In [ ]:
test_allx[is.na(Nom)]

### Disease-weeks reported by zone de santé

In [ ]:
combo <- merge(combo, zs_dict, by = c('ZS', 'PROV'), all.x = T)

In [ ]:
ggplot(aes(N), data =  combo[, .N, by = .(Nom, PROVINCE)]) +
    geom_histogram(bins = 40)

## Imputing 0s for non-reported diseases (from reported zone-weeks)

In [ ]:
# merge all data on rectangularized dataset

# combo[, merge_ID := paste0(Nom, "_", PROVINCE)]

ge <- expand.grid(
    zone_id_DHIS2 = unique(combo$zone_id_DHIS2), year = unique(combo$year),
    NUMSEM = unique(combo$NUMSEM), MALADIE = unique(combo$MALADIE)
)
setDT(ge)

combo <- merge(ge, combo, by = c("zone_id_DHIS2", "year", "NUMSEM", "MALADIE"), all.x = T)

In [ ]:
# compute number of non-NA observations by week
zone_weeks <- combo[, lapply(.SD, function(x) sum(!is.na(x))),
    by = .(zone_id_DHIS2, NUMSEM, year), .SDcols = "ZS"
]
zone_weeks[, reported := as.logical(ZS)]

In [ ]:
combo <- merge(combo, zone_weeks[, .(zone_id_DHIS2, NUMSEM, year, reported)],
    by = c("zone_id_DHIS2", "year", "NUMSEM"), all.x = TRUE
)

combo <- combo[reported == TRUE]

combo[, imputed := FALSE][is.na(Nom) & reported, imputed := TRUE]

In [ ]:
combo[reported == TRUE, TOTALCAS := rowSums(.SD, na.rm = T),
  .SDcols = c("C011MOIS", "C1259MOIS", "C5ANSP")
]
combo[reported == TRUE, TOTALDECES := rowSums(.SD, na.rm = T),
  .SDcols = c("D011MOIS", "D1259MOIS", "D5ANSP")
]

combo[, Nom := na.locf0(Nom, fromLast = TRUE), by = zone_id_DHIS2][, Nom := na.locf0(Nom), by = zone_id_DHIS2]
combo[, PROVINCE := na.locf0(PROVINCE,
  fromLast = TRUE
),
by = zone_id_DHIS2
][, PROVINCE := na.locf0(PROVINCE), by = zone_id_DHIS2]
combo[imputed == TRUE, C011MOIS := nafill(C011MOIS, fill = 0, nan = NA)]
combo[imputed == TRUE, D011MOIS := nafill(D011MOIS, fill = 0, nan = NA)]
combo[imputed == TRUE, C1259MOIS := nafill(C1259MOIS, fill = 0, nan = NA)]
combo[imputed == TRUE, D1259MOIS := nafill(D1259MOIS, fill = 0, nan = NA)]
combo[imputed == TRUE, C5ANSP := nafill(C5ANSP, fill = 0, nan = NA)]
combo[imputed == TRUE, D5ANSP := nafill(D5ANSP, fill = 0, nan = NA)]
combo[imputed == TRUE, TOTALCAS := nafill(TOTALCAS, fill = 0, nan = NA)]
combo[imputed == TRUE, TOTALDECES := nafill(TOTALDECES, fill = 0, nan = NA)]

In [ ]:
out <- combo[MALADIE == 'PALUDISME']

out <- out[, .(C011MOIS = suma(C011MOIS),
                   D011MOIS = suma(D011MOIS),
                   C1259MOIS = suma(C1259MOIS),
                   D1259MOIS = suma(D1259MOIS),
                   C5ANSP = suma(C5ANSP),
                   D5ANSP = suma(D5ANSP),
                   TOTALCAS = suma(TOTALCAS),
                   TOTALDECES = suma(TOTALDECES)),
              by = .(zone_id_DHIS2, year, NUMSEM, MALADIE)]

In [ ]:
cases <- melt(out,
  id.vars = c("zone_id_DHIS2", "year", "MALADIE", "NUMSEM"),
  measure.vars = c("C011MOIS", "C1259MOIS", "C5ANSP")
)

In [ ]:
names(cases)[5:6] <- c("Cas_Age", "Cas")

In [ ]:
deaths <- melt(out,
  id.vars = c("zone_id_DHIS2", "year", "MALADIE", "NUMSEM"),
  measure.vars = c("D011MOIS", "D1259MOIS", "D5ANSP")
)
names(deaths)[5:6] <- c("Deces_Age", "Deces")

In [ ]:
out <- cbind(cases, deaths[, 5:6])

### Organize data for export

In [ ]:
zones.DHIS <- read_sf(
    paste0(
        base_path,
        'geodata/org_units-2021-10-21-14-52_level_Zone_de_Sante.gpkg'
    )
)

In [ ]:
setDT(zones.DHIS)
zones.DHIS <- zones.DHIS[, .(Nom = name,
                             zone_id_DHIS2 = ref, 
                             PROVINCE = parent,
                             province_id_DHIS2 = parent_ref)]
zones.DHIS[, PROVINCE := gsub(' \\(Province\\)', '', PROVINCE)]

In [ ]:
out <- merge(out, zones.DHIS, by = 'zone_id_DHIS2')

In [ ]:
out[, Date := as.Date(paste(year, NUMSEM, 1, sep="-"), "%Y-%U-%u")]

In [ ]:
out <- out[Date < Sys.Date()]

In [ ]:
nrow(out)

In [ ]:
head(out)

In [ ]:
out_histo <- out[year < max(year)]
out_new <- out[year == max(year)]

### Output data

In [ ]:
# dbWriteTable(con, "DSE_palu_for_PNLP", out_histo, overwrite = TRUE)
# print("Upload complete")
# dbWriteTable(con, "DSE_palu_for_PNLP_2022", out_new, overwrite = TRUE)
# print("Upload complete")
# dbWriteTable(con, "DSE_palu_for_PNLP_tot", out, overwrite = TRUE)
# print("Upload complete")